In [6]:
pip install networkx

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [9]:
import warnings; warnings.filterwarnings('ignore')
import json, numpy as np, pandas as pd, networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split

np.random.seed(42)

DAG_PATH  = "../Génération de données synthétiques/LLM/dag_softmax.json"
DATA_PATH = "../Analyses_descriptives/Fraud Detection Dataset.csv"
OUTPUT    = "fraudes_synthetiques_dag.csv"
N         = 15_000    # fraudes à générer
BATCH     = 50_000   # taille du batch

dag   = json.load(open(DAG_PATH, encoding="utf-8"))
NODES = dag["noeuds"]; TARGET = "Fraudulent"
CONT  = ["Transaction_Amount","Time_of_Transaction","Account_Age",
         "Number_of_Transactions_Last_24H","Previous_Fraudulent_Transactions"]
CAT   = ["Transaction_Type","Device_Used","Location","Payment_Method"]

df = pd.read_csv(DATA_PATH)[NODES].dropna(subset=[TARGET])
print(f"Data: {len(df):,}  fraude={df[TARGET].mean():.2%}")

G = nx.DiGraph(); G.add_nodes_from(NODES)
for l in dag["liens"]: G.add_edge(l["de"], l["vers"])
parents = {n: list(G.predecessors(n)) for n in G.nodes()}
topo    = list(nx.topological_sort(G))

le_dict = {}; df_enc = df.copy()
for col in CAT:
    le = LabelEncoder(); mask = df_enc[col].notna()
    le.fit(df_enc.loc[mask, col].astype(str))
    codes = le.transform(df_enc.loc[mask, col].astype(str)).astype(float)
    df_enc[col] = np.nan; df_enc[col] = df_enc[col].astype(float)
    df_enc.loc[mask, col] = codes; le_dict[col] = le

def make_X(src, cols):
    X = src[cols].copy().apply(pd.to_numeric, errors="coerce").values.astype(float)
    for j in range(X.shape[1]):
        m = np.nanmedian(X[:, j])
        X[np.isnan(X[:, j]), j] = m if not np.isnan(m) else 0.
    return X

def youden(yt, yp):
    fpr, tpr, th = roc_curve(yt, yp)
    return float(th[np.argmax(tpr - fpr)])


models = {}; scalers = {}; seuil_youden = None
for node in topo:
    p = parents[node]
    if not p: continue
    vt = "binary" if node == TARGET else "categorical"
    mask = df_enc[node].notna()
    X = make_X(df_enc[mask], p)
    Y = df_enc.loc[mask, node].values.astype(float)
    Xtr, Xv, Ytr, Yv = train_test_split(X, Y, test_size=0.2, random_state=42)
    sc = StandardScaler()
    Xtr_sc = sc.fit_transform(Xtr); Xv_sc = sc.transform(Xv)
    scalers[node] = sc
    if vt == "binary":
        m = LogisticRegression(C=10, max_iter=2000)
        m.fit(Xtr_sc, Ytr)
        yp = m.predict_proba(Xv_sc)[:, 1]
        s  = youden(Yv, yp); seuil_youden = s
        print(f"  {node}: Seuil Youden={s:.4f}")
    else:
        m = LogisticRegression(C=1, max_iter=2000)
        m.fit(Xtr_sc, Ytr)
        print(f"  {node}: acc={m.score(Xv_sc, Yv):.4f}")
    models[node] = m



def encode_col(vals, col):
    """Encode une colonne catégorielle en codes numériques (vectorisé)."""
    le = le_dict[col]
    vals = np.array(vals).astype(str)
    known = np.isin(vals, le.classes_)
    codes = np.zeros(len(vals))
    codes[known]  = le.transform(vals[known])
    codes[~known] = 0
    return codes.astype(float)

def sample_categ(proba, n_classes):
    """Tirage multinomial vectorisé — remplace le for loop."""
    cumul = np.cumsum(proba, axis=1)
    r     = np.random.rand(len(proba), 1)
    return (r > cumul).sum(axis=1).clip(0, n_classes - 1)

all_rows = []; it = 0
while len(all_rows) < N and it < 30:
    it += 1; batch = {}

    # Racines
    for node in topo:
        if parents[node]: continue
        s2 = df[node].dropna()
        if node in CONT:
            batch[node] = s2.sample(BATCH, replace=True, random_state=it).values
        else:
            freq = s2.astype(str).value_counts(normalize=True)
            batch[node] = np.random.choice(freq.index, BATCH, p=freq.values)

    # Enfants catégoriels
    for node in topo:
        if node in batch or node == TARGET: continue
        cols2 = [encode_col(batch[par], par) if par in CAT
                 else batch[par].astype(float)
                 for par in parents[node]]
        Xs = np.column_stack(cols2)
        nn = np.where(np.isnan(Xs)); Xs[nn] = np.nanmedian(Xs, axis=0)[nn[1]]
        pr = models[node].predict_proba(scalers[node].transform(Xs))
        # Tirage vectorisé (pas de for loop !)
        codes = sample_categ(pr, pr.shape[1])
        batch[node] = le_dict[node].inverse_transform(codes.astype(int))

    # Fraudulent
    cols3 = [encode_col(batch[par], par) if par in CAT
             else batch[par].astype(float)
             for par in parents[TARGET]]
    Xf = np.column_stack(cols3)
    nn = np.where(np.isnan(Xf)); Xf[nn] = np.nanmedian(Xf, axis=0)[nn[1]]
    yp = models[TARGET].predict_proba(scalers[TARGET].transform(Xf))[:, 1]
    mf = yp >= seuil_youden

    if mf.sum() > 0:
        feats = [n for n in NODES if n != TARGET]
        db = pd.DataFrame({c: batch[c][mf] for c in feats})
        db[TARGET] = 1; all_rows.append(db)

    ntot = sum(len(d) for d in all_rows)
    print(f"  Batch {it}: gardé={mf.sum():,} ({mf.mean()*100:.1f}%)  total={ntot:,}/{N}")
    if ntot >= N: break

df_synth = pd.concat(all_rows, ignore_index=True).head(N)
df_synth[TARGET] = 1
df_synth.to_csv(OUTPUT, index=False)
print(df_synth.head(3).to_string(index=False))

Data: 51,000  fraude=4.92%
  Device_Used: acc=0.3187
  Fraudulent: Seuil Youden=0.0461
  Batch 1: gardé=38,539 (77.1%)  total=38,539/15000
 Transaction_Amount Transaction_Type  Time_of_Transaction Device_Used Location  Previous_Fraudulent_Transactions  Account_Age  Number_of_Transactions_Last_24H Payment_Method  Fraudulent
            1800.11    Bank Transfer                 11.0     Desktop    Miami                                 0           48                               12    Net Banking           1
            1752.48  Online Purchase                 11.0      Mobile  Chicago                                 2           37                                4    Net Banking           1
            2177.41      POS Payment                  5.0      Tablet New York                                 1            6                                1     Debit Card           1
